In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import argparse

parser = argparse.ArgumentParser(description="Plot GSEA results")
parser.add_argument("--gsea_results_pathways", type=str, required=True, help="Path to GSEA results for pathways")
parser.add_argument("--gsea_results_modules", type=str, required=True, help="Path to GSEA results for modules")
parser.add_argument("--filter_terms", type=str, required=True, help="Path to CSV file with terms to filter out")
args = parser.parse_args()

gsea_results_pathways = pd.read_csv(args.gsea_results_pathways)
gsea_results_modules = pd.read_csv(args.gsea_results_modules)
filter_terms = pd.read_csv(args.filter_terms)

def plot_gsea_results(results_df, term_type):
    """Plot the top enriched pathways/modules with color based on FDR q-value."""
    top_terms = results_df[["Term", "NES", "FDR q-val"]].sort_values("NES", ascending=False)

    bar_thickness = 0.4
    num_bars = len(top_terms)
    fig_height = bar_thickness * num_bars + 1.2

    fig, ax = plt.subplots(figsize=(10, fig_height))
    
    # Normalize FDR q-value for color mapping
    norm = plt.Normalize(vmin=0, vmax=0.05)
    
    # Create a colormap from blue to purple to red
    blue_purple_red = LinearSegmentedColormap.from_list("red_purple_blue", ["red", "purple", "blue"])
    cmap = blue_purple_red
    #cmap = plt.cm.plasma 
    bar_colors = cmap(norm(top_terms["FDR q-val"].values))

    # Create barplot
    sns.barplot(data=top_terms, x="NES", y="Term", palette=bar_colors, ax=ax,
    )

    ax.set_xlabel("Normalized Enrichment Score (NES)", fontsize=16)
    ax.set_ylabel(term_type.capitalize(), fontsize=16)
    #ax.set_title(f"Enriched KEGG {term_type} in GSEA (FDR < 0.1)")
    
    # Set tick parameters
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.spines['top'].set_linewidth(1.2)
    ax.spines['right'].set_linewidth(1.2)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)
    
    ax.set_ylim(-0.5, num_bars - 0.5)  # tighten y-axis limits

    # Create colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    # Shrink colorbar and reverse direction (high FDR at bottom)
    cbar = fig.colorbar(sm, ax=ax, aspect=20, shrink=0.45)
    cbar.ax.invert_yaxis()  # invert the colorbar to show low FDR at the top
    cbar.set_label("FDR q-value", fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    plt.tight_layout()
    plt.savefig(snakemake.output[f"gsea_plot_{term_type}"], dpi=300, transparent=True)
    plt.close()


# Plot results
plot_gsea_results(gsea_results_pathways, "pathways")
plot_gsea_results(gsea_results_modules, "modules")